# Lab 09-06 · PN Junction I–V, Capacitance and Breakdown

Companion to chapter [09-06](../../chapters/09-06-pn-junction-iv-and-capacitance.md). Handout: [README.md](README.md).

The diode throughout is the chapter's P⁺N junction: $N_A$ = 10¹⁹ cm⁻³, $N_D$ = 10¹⁶ cm⁻³, $\tau_p$ = 1 µs, area 10⁻⁴ cm², at 300 K.

In [1]:
import math
import numpy as np
from scipy.integrate import solve_ivp, cumulative_trapezoid, trapezoid
from scipy.linalg import solve_banded
from scipy.optimize import brentq, curve_fit
from scipy.special import erf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Q = 1.602176634e-19
KB = 1.380649e-23 / Q                  # eV/K
VT = KB * 300
NI = 1e10
EPS = 11.7 * 8.8541878128e-14            # F/cm
NA, ND, TP, AREA = 1e19, 1e16, 1e-6, 1e-4
mup = lambda N: 48 + 402 / (1 + (N / 6.3e16) ** 0.76)
mun = lambda N: 92 + 1268 / (1 + (N / 1.3e17) ** 0.91)
DP = VT * mup(ND); LP = math.sqrt(DP * TP)
taun = 1 / (0.3e-30 * NA**2); DN = VT * mun(NA); LN = math.sqrt(DN * taun)
JP0 = Q * NI**2 * DP / (LP * ND); JN0 = Q * NI**2 * DN / (LN * NA)
IS = AREA * (JP0 + JN0)
print(f"D_p = {DP:.4f} cm^2/s, L_p = {LP*1e4:.2f} um, I_s = {IS:.4e} A")
assert abs(IS - 4.973e-16) / 4.973e-16 < 2e-4

D_p = 9.5756 cm^2/s, L_p = 30.94 um, I_s = 4.9731e-16 A


## Part 1 — A real diode, part by part

The model has four parts (chapter §3.5):

- **Diffusion** current, with high-level injection: the edge hole density solves $p(N_D + p) = n_i^2e^{V_j/V_T}$, which reduces to the law of the junction at low injection.
- **Depletion-region recombination**, $I_{r0}(e^{V_j/2V_T} - 1)$, with $W$ fixed at its 0.4 V value as in Example 09-06.3.
- **Series resistance**: the terminal voltage is $V = V_j + IR_s$.

*Predict first:* with $R_s$ = 50 Ω, at what current does $IR_s$ equal $V_T$?

In [2]:
VBI = VT * math.log(NA * ND / NI**2)
W04 = math.sqrt(2 * EPS * (VBI - 0.4) * (1 / NA + 1 / ND) / Q)
TAU0, RS = 1e-6, 50.0

def i_parts(vj):
    pe = (-ND + np.sqrt(ND**2 + 4 * NI**2 * np.exp(vj / VT))) / 2
    idiff = AREA * Q * DP / LP * (pe - NI**2 / ND) * (1 + JN0 / JP0)
    irec = AREA * Q * NI * W04 / (2 * TAU0) * (np.exp(vj / (2 * VT)) - 1)
    return idiff, irec

vj = np.linspace(0.05, 0.95, 3000)
idiff, irec = i_parts(vj)
itot = idiff + irec
vterm = vj + itot * RS
eta = np.gradient(vterm, np.log(itot), edge_order=2) / VT
i_rs = VT / RS
print(f"I R_s = V_T at I = {i_rs*1e3:.3f} mA")
k = np.argmin(eta[(vterm > 0.4) & (vterm < 0.7)])
print(f"minimum local ideality factor {eta[(vterm > 0.4) & (vterm < 0.7)][k]:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].semilogy(vterm, itot, "k"); ax[0].set_xlim(0, 0.9); ax[0].set_ylim(1e-13, 1e-1)
ax[0].set_xlabel("terminal voltage (V)"); ax[0].set_ylabel("I (A)")
ax[1].plot(vterm, eta); ax[1].set_xlim(0, 0.9); ax[1].set_ylim(0.8, 3); ax[1].set_xlabel("V"); ax[1].set_ylabel("eta")
plt.tight_layout(); plt.show()
assert abs(i_rs * 1e3 - 0.517) < 5e-4

I R_s = V_T at I = 0.517 mA
minimum local ideality factor 1.038


**Fitting a measurement.** Synthetic "measured" data come from a compact model $I = I_0(e^{(V - IR_s)/\eta V_T} - 1)$ with known parameters plus 1% random noise. A fit should recover the parameters. Because the model is implicit in $I$, fit $V(I) = \eta V_T\ln(I/I_0 + 1) + IR_s$ instead.

In [3]:
rng = np.random.default_rng(1)
I0_true, eta_true, rs_true = 5e-16, 1.05, 50.0
I_meas = np.geomspace(1e-9, 2e-3, 40)
V_meas = eta_true * VT * np.log(I_meas / I0_true + 1) + I_meas * rs_true
I_noisy = I_meas * (1 + 0.01 * rng.standard_normal(I_meas.size))
model = lambda I, lnI0, eta_, rs_: eta_ * VT * np.log(I / np.exp(lnI0) + 1) + I * rs_
popt, _ = curve_fit(model, I_noisy, V_meas, p0=(math.log(1e-15), 1.0, 10.0))
I0_fit, eta_fit, rs_fit = math.exp(popt[0]), popt[1], popt[2]
print(f"fit: I0 = {I0_fit:.3e} A, eta = {eta_fit:.4f}, Rs = {rs_fit:.2f} ohm")
assert abs(eta_fit - eta_true) < 0.01 and abs(rs_fit - rs_true) < 2.5

fit: I0 = 5.018e-16 A, eta = 1.0502, Rs = 50.03 ohm


## Part 2 — Temperature and a PTAT thermometer

Hold $D_p$ and $\tau_p$ fixed, so $I_s \propto n_i^2$ with the course model. *Predict first:* is $|dV/dT|$ larger at 1 µA or at 1 mA?

In [4]:
def Eg(T): return 1.17 - 4.73e-4 * T**2 / (T + 636)
def niT(T): return NI * (T / 300)**1.5 * math.exp(Eg(300) / (2 * KB * 300) - Eg(T) / (2 * KB * T))
def V_at(I, T): return KB * T * math.log(I / (IS * (niT(T) / NI)**2) + 1)

Ts = np.linspace(250, 400, 61)
for I in (1e-6, 1e-5, 1e-3):
    dvdt = (V_at(I, 300.001) - V_at(I, 299.999)) / 0.002
    print(f"I = {I:.0e} A: V(300 K) = {V_at(I, 300):.4f} V, dV/dT = {dvdt*1e3:.3f} mV/K")
    plt.plot(Ts, [V_at(I, T) for T in Ts], label=f"{I:.0e} A")
plt.xlabel("T (K)"); plt.ylabel("V (V)"); plt.legend(); plt.show()
d10 = (V_at(1e-5, 300.001) - V_at(1e-5, 299.999)) / 0.002
assert abs(d10 * 1e3 - (-2.217)) < 5e-4
# PTAT thermometer: two currents a decade apart; T recovered from Delta V alone
for T in (250, 300, 350, 400):
    dV = V_at(1e-4, T) - V_at(1e-5, T)
    print(f"T = {T} K: Delta V = {dV*1e3:.3f} mV -> T_measured = {dV / (KB * math.log(10)):.2f} K")

I = 1e-06 A: V(300 K) = 0.5538 V, dV/dT = -2.416 mV/K
I = 1e-05 A: V(300 K) = 0.6133 V, dV/dT = -2.217 mV/K
I = 1e-03 A: V(300 K) = 0.7324 V, dV/dT = -1.820 mV/K
T = 250 K: Delta V = 49.605 mV -> T_measured = 250.00 K
T = 300 K: Delta V = 59.526 mV -> T_measured = 300.00 K
T = 350 K: Delta V = 69.447 mV -> T_measured = 350.00 K
T = 400 K: Delta V = 79.368 mV -> T_measured = 400.00 K


## Part 3 — Reverse recovery from the diffusion equation

In dimensionless units ($x$ in $L_p$, $t$ in $\tau_p$): $u_t = u_{xx} - u$, starting from $u = e^{-x}$. At $t = 0$ the flux at $x = 0$ reverses to $I_R/I_F$ times its forward value. Crank–Nicolson time steps with a ghost node impose the flux condition. The storage time is when $u(0, t) = 0$.

*Predict first (the chapter's guided exercise):* doubling $I_R$ from $I_F$ to $2I_F$ — does the exact storage time halve, fall by more, or by less? And charge control?

In [5]:
def storage(ratio, L=12.0, n=2401, dt=1e-5):
    x = np.linspace(0, L, n); h = x[1] - x[0]; N = n - 1
    main = np.full(N, -2.0 / h**2 - 1.0)
    up = np.full(N - 1, 1.0 / h**2); up[0] = 2.0 / h**2
    lo = np.full(N - 1, 1.0 / h**2)
    b = np.zeros(N); b[0] = -2 * ratio / h
    A1 = np.zeros((3, N)); A1[0, 1:] = -0.5 * dt * up; A1[1, :] = 1 - 0.5 * dt * main; A1[2, :-1] = -0.5 * dt * lo
    u = np.exp(-x[:N]); t = 0.0
    while True:
        rhs = u + 0.5 * dt * (main * u + np.concatenate([up * u[1:], [0]]) + np.concatenate([[0], lo * u[:-1]])) + dt * b
        new = solve_banded((1, 1), A1, rhs); t += dt
        if new[0] <= 0:
            return t - dt + dt * u[0] / (u[0] - new[0])
        u = new

for r in (0.5, 1.0, 2.0):
    ts = storage(r)
    king = brentq(lambda s: erf(math.sqrt(s)) - 1 / (1 + r), 1e-9, 10)
    print(f"I_R/I_F = {r}: PDE t_s/tau = {ts:.4f}, Kingston {king:.4f}, charge control {math.log(1 + 1/r):.4f}")
ts1, ts2 = storage(1.0), storage(2.0)
print(f"doubling I_R: exact factor {ts1/ts2:.3f}, charge-control factor {math.log(2)/math.log(1.5):.3f}")
assert abs(ts1 - 0.2275) < 1e-4 and abs(ts2 - 0.0928) < 1e-4
assert abs(ts1 / ts2 - 2.45) < 0.005 and abs(math.log(2) / math.log(1.5) - 1.71) < 0.005

I_R/I_F = 0.5: PDE t_s/tau = 0.4680, Kingston 0.4680, charge control 1.0986


I_R/I_F = 1.0: PDE t_s/tau = 0.2275, Kingston 0.2275, charge control 0.6931


I_R/I_F = 2.0: PDE t_s/tau = 0.0928, Kingston 0.0928, charge control 0.4055


doubling I_R: exact factor 2.452, charge-control factor 1.710


## Part 4 — Avalanche: the multiplication equations

Integrate the current-continuity equation $dJ_n/dx = \alpha_nJ_n + \alpha_pJ_p$ (with $J_n + J_p = J$) through a one-sided junction's triangular field. Starting from electrons, $J_n(0)/J = 1/M_n$. Starting from holes, $J_p(W)/J = 1/M_p$. Breakdown is where either reaches zero.

*Predict first:* by how much does $M$ rise between 80% and 95% of $V_B$?

In [6]:
an = lambda F: 3.318e5 * np.exp(-1.174e6 / F)
ap = lambda F: np.where(F < 5.3e5, 2e6 * np.exp(-1.97e6 / F), 5.6e5 * np.exp(-1.32e6 / F))

def inv_M(Em, N, hole=False):
    W = EPS * Em / (Q * N)
    F = lambda x: max(Em * (1 - x / W), 1e3)
    if not hole:   # electrons enter at x = 0; integrate back from Jn(W) = J = 1
        rhs = lambda x, y: [float(an(F(x))) * y[0] + float(ap(F(x))) * (1 - y[0])]
        return solve_ivp(rhs, [W, 0], [1.0], rtol=1e-10, atol=1e-13, max_step=W / 400).y[0, -1]
    rhs = lambda x, y: [-(float(an(F(x))) * (1 - y[0]) + float(ap(F(x))) * y[0])]
    return solve_ivp(rhs, [0, W], [1.0], rtol=1e-10, atol=1e-13, max_step=W / 400).y[0, -1]

def VB(N, hole=False):
    Em = brentq(lambda E: inv_M(E, N, hole), 1.5e5, 9e5, xtol=1.0)
    W = EPS * Em / (Q * N)
    return Em, Em * W / 2 - VT * math.log(1e19 * N / NI**2), W

for N in (1e15, 3e15, 1e16, 3e16, 1e17):
    Em, V, W = VB(N)
    print(f"N = {N:.0e}: peak field {Em:.3e} V/cm, V_B = {V:.1f} V, W = {W*1e4:.2f} um, Hu's rule {15*(1e17/N)**0.6:.1f} V")
Ve = VB(1e16)[1]; Vh = VB(1e16, hole=True)[1]
print(f"1e16: electron-initiated {Ve:.2f} V, hole-initiated {Vh:.2f} V")
assert abs(Ve - 54.7) < 0.05 and abs(Vh - 54.7) < 0.05

def M_at(VR, N=1e16):
    Vt = VT * math.log(1e19 * N / NI**2) + VR
    W = math.sqrt(2 * EPS * Vt / (Q * N)); Em = 2 * Vt / W
    return 1 / inv_M(Em, N)
VRs = np.linspace(5, 54.5, 60)
plt.semilogy(VRs, [M_at(v) for v in VRs]); plt.xlabel("reverse bias (V)"); plt.ylabel("M_n"); plt.show()
print(f"M at 0.80 V_B: {M_at(0.80*Ve):.2f};  M at 0.95 V_B: {M_at(0.95*Ve):.2f}")

N = 1e+15: peak field 2.852e+05 V/cm, V_B = 262.1 V, W = 18.44 um, Hu's rule 237.7 V


N = 3e+15: peak field 3.370e+05 V/cm, V_B = 121.5 V, W = 7.26 um, Hu's rule 123.0 V


N = 1e+16: peak field 4.146e+05 V/cm, V_B = 54.7 V, W = 2.68 um, Hu's rule 59.7 V


N = 3e+16: peak field 5.154e+05 V/cm, V_B = 27.7 V, W = 1.11 um, Hu's rule 30.9 V


N = 1e+17: peak field 7.021e+05 V/cm, V_B = 15.0 V, W = 0.45 um, Hu's rule 15.0 V


1e16: electron-initiated 54.68 V, hole-initiated 54.68 V


M at 0.80 V_B: 2.53;  M at 0.95 V_B: 9.12


## Questions

Answer the discussion questions in the [lab handout](README.md). Then try the optional extension: replace the long region in Part 3 with a short one (absorbing boundary at $x = W_n/L_p$) and compare the exact and charge-control storage times.